<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/sankey2_master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sankey diagram
### Target Group → Target → Land Use

#Import libaries

In [ ]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb
import string
import plotly.colors

In [ ]:
# Airtable credentials
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'

In [ ]:
# Helper function to fetch data from Airtable
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [ ]:
# Fetch data
tabel0 = fetch_airtable_data("tblzHR1WHYHA5MlwQ")  # Policy Source
tabel1 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets (mellemtabel)
tabel2 = fetch_airtable_data("tblVarbVYd96JUE6f")  # Target Group
tabel3 = fetch_airtable_data("tbl7OYOXduME11uh7")   # Slut-targets

In [ ]:
# Explode and rename
t0 = tabel0.copy().explode("Targets (policy targets)").rename(columns={"Targets (policy targets)": "target_id"})
t1 = tabel1.copy().explode("Target Group").rename(columns={"Target Group": "target_group_id"})
t2 = tabel2.copy().explode("Targets").rename(columns={"Targets": "target_id"})

In [ ]:
# Register in DuckDB
duckdb.register("tabel0", t0)
duckdb.register("tabel1_exp", t1)
duckdb.register("tabel2_exp", t2)
duckdb.register("tabel3", tabel3)

In [ ]:
# SQL query
query = """
SELECT
    t0."Policy source"       AS policy_source,
    t1."Target name"         AS mid_target,
    t2."Target Group"        AS target_group,
    t3."Target name"         AS final_target
FROM
    tabel0 t0
JOIN
    tabel1_exp t1 ON t0.target_id = t1.id
JOIN
    tabel2_exp t2 ON t1.target_group_id = t2.id
JOIN
    tabel3 t3 ON t2.target_id = t3.id
"""

results = duckdb.sql(query).df()


In [ ]:
# Forkort og saml labels
results['policy_source'] = results['policy_source'].apply(lambda x: x[:70] + '…' if isinstance(x, str) and len(x) > 70 else x)
results['final_target'] = results['final_target'].apply(lambda x: x[:40] + '…' if isinstance(x, str) and len(x) > 50 else x)
source_labels = results['policy_source']
source_labels = results['policy_source']
middle_labels = results['target_group']
target_labels = results['final_target']

all_labels = pd.concat([source_labels, middle_labels, target_labels])
unique_labels = pd.unique(all_labels)
label_to_index = {label: i for i, label in enumerate(unique_labels)}

In [ ]:
# Forbindelser
links1 = pd.DataFrame({
    'source': source_labels.map(label_to_index),
    'target': middle_labels.map(label_to_index),
    'value': 1
})
links2 = pd.DataFrame({
    'source': middle_labels.map(label_to_index),
    'target': target_labels.map(label_to_index),
    'value': 1
})

all_links = pd.concat([links1, links2])

In [ ]:
# Brug Plotlys palette
node_colors = plotly.colors.qualitative.Plotly

# Tildel farver til unikke labels (noder)
color_map = {label: node_colors[i % len(node_colors)] for i, label in enumerate(unique_labels)}

# Liste med farver i samme rækkefølge som unique_labels
node_colors_list = [color_map[label] for label in unique_labels]


# Funktion til at lysne farver (uden brug af gennemsigtighed)
def lighten(hex_color, factor=0.5):
    from plotly.colors import hex_to_rgb
    r, g, b = hex_to_rgb(hex_color)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'rgb({r},{g},{b})'

# Lysnet farve til hver link ud fra source-node
link_colors = [lighten(node_colors_list[src], factor=0.8) for src in all_links['source']]



# Trin 3: Sankey-diagram
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=60,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels),
        color=node_colors_list
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['value'],
        color=link_colors
    )
)])
fig.update_layout(title_text="Policy Source → Target Group → Target", font_size=12, height=6000)
fig.show()

In [ ]:
# Gem som interaktiv HTML-fil
# fig.write_html("sankey_diagram_filtered.html")

# Download i Colab
# from google.colab import files
# files.download("sankey_diagram_filtered.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>